# Amazon S3 — Day-to-Day Operations with Python

**VidyaSankalp · AWS Series · Module 3 notebook** · Tutor: Prudhvi Akella

```
 1 Setup → 2 Create bucket → 3 Upload → 4 List → 5 Download → 6 Read → 7 Update → 8 Copy / rename
        → 9 Share link → 10 Delete → 11 Clean up
```

**Before you start:** run `aws configure` once (Module 1, "Before you start"). Python uses the same login.
Run the cells top to bottom.

## 1. Setup

```
 notebook ──> boto3 ──> your AWS login (from aws configure) ──> S3 in us-east-1
```

`boto3` is AWS's Python library. `s3` below is the object we use for every S3 action.

In [ ]:
# %pip install boto3 pandas      # run once if not installed

import json
from pathlib import Path

import boto3

REGION = "us-east-1"
s3 = boto3.client("s3", region_name=REGION)

Bucket names must be unique worldwide, so we add your 12-digit AWS account ID to the name.

In [ ]:
account_id = boto3.client("sts").get_caller_identity()["Account"]

BUCKET = f"vs-lab-notebook-{account_id}"
print("Bucket:", BUCKET)

LOCAL = Path("s3_lab_files")          # a folder on your computer for sample files and downloads
LOCAL.mkdir(exist_ok=True)

## 2. Create a bucket

```
 create_bucket ──> new bucket: empty and private
```

In [ ]:
s3.create_bucket(Bucket=BUCKET)

for b in s3.list_buckets()["Buckets"]:
    print(b["Name"])

## 3. Upload files

```
 file on your computer ──> upload_file(file, bucket, key) ──> S3
 Python data            ──> put_object(bucket, key, body)  ──> S3
```

The **key** is the file's full name inside the bucket, e.g. `raw/hr/leave-policy.json`.
The part before the last `/` shows up as "folders" in the console.

First, create three sample files on your computer:

In [ ]:
leave_policy = {"policy_id": "HR-001", "title": "Leave policy", "annual_leave_days": 24, "updated_by": "anita"}
(LOCAL / "leave-policy.json").write_text(json.dumps(leave_policy, indent=2))

(LOCAL / "employees.csv").write_text(
    "emp_id,name,department,city\n"
    "E1001,Anita,HR,Hyderabad\n"
    "E1002,Ravi,Finance,Mumbai\n"
    "E1003,Sana,IT,Pune\n"
)

(LOCAL / "travel-policy.txt").write_text("Economy class for trips under 6 hours.\n")

print([f.name for f in LOCAL.iterdir()])

Upload one file at a time:

In [ ]:
s3.upload_file(str(LOCAL / "leave-policy.json"), BUCKET, "raw/hr/leave-policy.json")
s3.upload_file(str(LOCAL / "employees.csv"),     BUCKET, "raw/hr/employees.csv")
s3.upload_file(str(LOCAL / "travel-policy.txt"), BUCKET, "raw/hr/travel-policy.txt")
print("Uploaded")

Upload Python data directly — no file needed:

In [ ]:
remote_policy = {"policy_id": "HR-002", "title": "Remote work", "days_per_week": 2}

s3.put_object(Bucket=BUCKET, Key="raw/hr/remote-policy.json", Body=json.dumps(remote_policy))
print("Uploaded")

Upload every file in a folder:

In [ ]:
for file in LOCAL.iterdir():
    if file.is_file():
        s3.upload_file(str(file), BUCKET, f"backup/{file.name}")
        print("Uploaded", file.name)

## 4. List files

```
 list_objects_v2(bucket, prefix) ──> the keys that start with prefix
```

Leave the prefix empty to list everything. (One call returns up to 1,000 files — plenty for learning.)

In [ ]:
def list_files(prefix=""):
    response = s3.list_objects_v2(Bucket=BUCKET, Prefix=prefix)
    return [obj["Key"] for obj in response.get("Contents", [])]   # no "Contents" when nothing matches

print("Everything:")
for key in list_files():
    print("  ", key)

print("Only raw/hr/:")
for key in list_files("raw/hr/"):
    print("  ", key)

Size and date of one file, without downloading it:

In [ ]:
info = s3.head_object(Bucket=BUCKET, Key="raw/hr/employees.csv")
print("Size:", info["ContentLength"], "bytes")
print("Last modified:", info["LastModified"])

## 5. Download a file

```
 S3 ──> download_file(bucket, key, file) ──> file on your computer
```

In [ ]:
s3.download_file(BUCKET, "raw/hr/employees.csv", str(LOCAL / "downloaded-employees.csv"))
print((LOCAL / "downloaded-employees.csv").read_text())

## 6. Read a file straight into Python (no download)

```
 get_object(bucket, key)["Body"].read() ──> bytes ──> text / JSON / table
```

This is what Lambdas and RAG pipelines do: read the file and use it, without saving it to disk.

JSON → Python dictionary:

In [ ]:
body = s3.get_object(Bucket=BUCKET, Key="raw/hr/leave-policy.json")["Body"].read()
policy = json.loads(body)

print(policy)
print("Annual leave days:", policy["annual_leave_days"])

CSV → table (pandas):

In [ ]:
import csv
import io

body = s3.get_object(
    Bucket=BUCKET,
    Key="raw/hr/employees.csv"
)["Body"]

employees = list(csv.DictReader(io.TextIOWrapper(body, encoding="utf-8")))
employees

Plain text:

In [ ]:
text = s3.get_object(Bucket=BUCKET, Key="raw/hr/travel-policy.txt")["Body"].read().decode("utf-8")
print(text)

## 7. Update a JSON file

S3 cannot change part of a file. To update, you **read → change → upload the whole file again** to the same key.

```
 get_object ──> change in Python ──> put_object (same key)
```

In [ ]:
policy = json.loads(s3.get_object(Bucket=BUCKET, Key="raw/hr/leave-policy.json")["Body"].read())

policy["annual_leave_days"] = 30
policy["updated_by"] = "ravi"

s3.put_object(Bucket=BUCKET, Key="raw/hr/leave-policy.json", Body=json.dumps(policy))

print(json.loads(s3.get_object(Bucket=BUCKET, Key="raw/hr/leave-policy.json")["Body"].read()))

## 8. Copy and rename

```
 copy   = copy_object (old key → new key)
 rename = copy_object + delete_object (old key)        S3 has no rename
```

In [ ]:
# Copy
s3.copy_object(Bucket=BUCKET, Key="processed/hr/leave-policy.json",
               CopySource={"Bucket": BUCKET, "Key": "raw/hr/leave-policy.json"})

# Rename: travel-policy.txt -> travel-policy-2026.txt
s3.copy_object(Bucket=BUCKET, Key="raw/hr/travel-policy-2026.txt",
               CopySource={"Bucket": BUCKET, "Key": "raw/hr/travel-policy.txt"})
s3.delete_object(Bucket=BUCKET, Key="raw/hr/travel-policy.txt")

for key in list_files():
    print(key)

## 9. Share a file with a link that expires

```
 generate_presigned_url ──> https://...  (anyone with the link can download that one file until it expires)
```

Share links must be signed the current way (`s3v4`), so we make a small client just for signing.

In [ ]:
from botocore.config import Config

signer = boto3.client("s3", region_name=REGION, config=Config(signature_version="s3v4"))

url = signer.generate_presigned_url(
    "get_object",
    Params={"Bucket": BUCKET, "Key": "raw/hr/leave-policy.json"},
    ExpiresIn=300,                     # 5 minutes
)
print(url)

## 10. Delete files

```
 one file            delete_object(bucket, key)
 several files       delete_objects(bucket, list of keys)
 a whole "folder"    list the keys under the prefix, then delete_objects
```

⚠️ There is no recycle bin. Deleted means gone.

One file:

In [ ]:
s3.delete_object(Bucket=BUCKET, Key="processed/hr/leave-policy.json")
print(list_files("processed/"))         # [] — gone

Several files:

In [ ]:
s3.delete_objects(Bucket=BUCKET, Delete={"Objects": [
    {"Key": "backup/employees.csv"},
    {"Key": "backup/travel-policy.txt"},
]})
print(list_files("backup/"))

Everything in a "folder":

In [ ]:
keys = list_files("backup/")
if keys:
    s3.delete_objects(Bucket=BUCKET, Delete={"Objects": [{"Key": k} for k in keys]})

print("Left in the bucket:", list_files())

## 11. Clean up

A bucket must be empty before you can delete it.

```
 delete all files ──> delete_bucket
```

In [ ]:
keys = list_files()
if keys:
    s3.delete_objects(Bucket=BUCKET, Delete={"Objects": [{"Key": k} for k in keys]})

s3.delete_bucket(Bucket=BUCKET)
print("Deleted bucket", BUCKET)

## Quick reference

| Task | Call |
|---|---|
| Create bucket | `s3.create_bucket(Bucket=...)` |
| Upload a file / Python data | `s3.upload_file(file, bucket, key)` / `s3.put_object(Bucket, Key, Body)` |
| List | `s3.list_objects_v2(Bucket, Prefix)` |
| Size and date | `s3.head_object(Bucket, Key)` |
| Download | `s3.download_file(bucket, key, file)` |
| Read into Python | `s3.get_object(Bucket, Key)["Body"].read()` |
| Copy | `s3.copy_object(Bucket, Key, CopySource)` |
| Share link | `generate_presigned_url("get_object", Params, ExpiresIn)` |
| Delete one / several | `s3.delete_object` / `s3.delete_objects` |
| Delete bucket | empty it, then `s3.delete_bucket(Bucket=...)` |

**Common errors**

| Error | Meaning |
|---|---|
| `NoCredentialsError` | `aws configure` not done — run it, then restart the notebook |
| `AccessDenied` | Your user is not allowed to do this |
| `NoSuchKey` | Wrong key — keys are case-sensitive and include the folder part |
| `BucketAlreadyExists` | Someone else in the world has that name |
| `BucketNotEmpty` | Delete the files first |